In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# Creating a sample dataset with 4 well-separated clusters
X, y = make_blobs(n_samples=800, n_features=3, centers=4, random_state=42)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=y, cmap="viridis", s=20)
ax.set_title("Synthetic 3-D blob dataset")
plt.show()

# ==========================================================
# Fitting KMeans with scikit-learn
# ==========================================================
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
kmeans = kmeans.fit(X)

# Getting the cluster labels
labels = kmeans.predict(X)

# Centroid values
centroids = kmeans.cluster_centers_

print("Centroid values (scikit-learn):")
print(centroids)


In [ ]:
# K-Means Clustering on the "cars" dataset
# Goal: cluster cars by their specs (mpg, cylinders, displacement, hp, weight,
# 0-60 time, year) and see how well the clusters line up with country of origin.

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans

# Importing the dataset (path is relative to this notebook's own folder)
dataset = pd.read_csv('data/cars.csv')
dataset.columns = [c.strip() for c in dataset.columns]
print(dataset.head())

# Separate the numeric features (drop the brand/origin label) from the target
X = dataset.iloc[:, :-1].copy()
X.columns = ['mpg', 'cylinders', 'cubicinches', 'hp', 'weightlbs', 'time-to-60', 'year']

# The numeric columns were parsed as strings in the raw CSV in a couple of places
# (stray spaces, blanks) -- coerce everything to numeric and fill gaps with the
# column mean, the modern replacement for the removed DataFrame.convert_objects().
X = X.apply(pd.to_numeric, errors="coerce")
print("\nMissing values per column before imputation:")
print(X.isnull().sum())
X = X.fillna(X.mean())

# Using the elbow method to find the optimal number of clusters
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=0)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 11), wcss, marker="o")
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS (within-cluster sum of squares)')
plt.show()

# The elbow sits around k=3, which conveniently matches the three regions
# (US / Japan / Europe) the "brand" column actually encodes.
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=300, n_init=10, random_state=0)
y_kmeans = kmeans.fit_predict(X)

# .as_matrix() was removed from pandas years ago -- .to_numpy() is the replacement
X_arr = X.to_numpy()

# Visualising the clusters on the first two features (mpg vs cylinders)
plt.figure(figsize=(7, 5))
colors = ['red', 'blue', 'green']
for cluster_id, color in enumerate(colors):
    mask = y_kmeans == cluster_id
    plt.scatter(X_arr[mask, 0], X_arr[mask, 1], s=60, c=color, label=f'cluster {cluster_id}')
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            s=250, c='gold', edgecolor='black', marker='X', label='centroids')
plt.xlabel('mpg'); plt.ylabel('cylinders')
plt.title('K-Means clusters of cars (by spec)')
plt.legend()
plt.show()


In [ ]:
# How well do the unsupervised clusters line up with the real country-of-origin label?
dataset['cluster'] = y_kmeans
print(pd.crosstab(dataset['brand'], dataset['cluster']))

print("\nEach cluster is dominated by one region, even though KMeans never saw the")
print("brand column -- engine size, weight and mpg alone are enough to separate")
print("US, Japanese and European cars into fairly clean groups.")
